# 02 - Baseline de ML clasico

Pipeline: extraemos features con un backbone preentrenado (ResNet18 frozen)
y entrenamos clasificadores clasicos de scikit-learn.

Etapas:
1. Setup, config y data loaders.
2. Extraccion de features con ResNet18 (sin entrenamiento).
3. Entrenamiento y comparacion de modelos sklearn.
4. Evaluacion final en validacion (accuracy, F1 macro, classification report).
5. Persistencia del mejor modelo y logging opcional a W&B.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.utils.config import load_yaml_config
from src.utils.reproducibility import set_global_seed

import joblib
import numpy as np
import torch
import wandb
from wandb.errors import UsageError
from torch import nn
from torchvision import models

from src.utils.wandb_utils import finish_wandb_run, init_wandb_run

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline_ml.yaml"
config = load_yaml_config(CONFIG_PATH)
set_global_seed(config["seed"])

configured_device = config.get("device", "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else configured_device)
config["runtime_device"] = str(device)

print(f"Configured device (yaml): {configured_device} | Runtime device: {device}")
if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available; using CPU.")

if wandb.run is not None:
    wandb.finish()

run = None
_wandb_init_attempted = False
wb_step = 0


def ensure_wandb_run() -> None:
    global run, _wandb_init_attempted
    if _wandb_init_attempted:
        return
    _wandb_init_attempted = True
    try:
        run = init_wandb_run(
            config=config,
            enabled=config["tracking"].get("use_wandb", False),
            project=config["tracking"]["project"],
            run_name=config["tracking"].get("run_name", config["experiment_name"]),
            tags=config["tracking"].get("tags"),
        )
    except Exception as error:
        print(f"W&B desactivado para esta ejecucion: {error}")
        run = None


def log_wandb(metrics: dict) -> None:
    global wb_step, run, _wandb_init_attempted
    ensure_wandb_run()
    if run is None:
        return
    try:
        wb_step += 1
        run.log({"progress_step": wb_step, **metrics})
    except UsageError:
        # Run closed (e.g., cell re-executed after finish). Re-open once and retry.
        run = None
        _wandb_init_attempted = False
        ensure_wandb_run()
        if run is None:
            return
        wb_step += 1
        run.log({"progress_step": wb_step, **metrics})


log_wandb({"phase": "setup", "runtime_device": str(device)})

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


Configured device (yaml): cpu | Runtime device: cuda
CUDA device name: NVIDIA GeForce RTX 4070 Laptop GPU


wandb: Currently logged in as: 202513902 (adne-image-classification) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Data loaders

In [2]:
import torch
from torch.utils.data import DataLoader

from src.data.dataset import load_imagefolder_datasets

data_root = PROJECT_ROOT / config["data"]["root_dir"]
image_size = config["data"]["image_size"]
batch_size = config["data"]["batch_size"]
num_workers = config["data"].get("num_workers", 0)

train_dataset, val_dataset = load_imagefolder_datasets(
    root_dir=data_root,
    train_subdir=config["data"].get("train_subdir", "train"),
    val_subdir=config["data"].get("val_subdir", "val"),
    image_size=image_size,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

class_names = train_dataset.classes
num_classes = len(class_names)
print({"train": len(train_dataset), "val": len(val_dataset), "num_classes": num_classes})
log_wandb(
    {
        "phase": "data_ready",
        "num_classes": num_classes,
        "num_train_samples": len(train_dataset),
        "num_val_samples": len(val_dataset),
    }
)


{'train': 189641, 'val': 33461, 'num_classes': 29}


## Extraccion de features con ResNet18 (frozen)

In [3]:
from time import perf_counter
import hashlib
import json

from tqdm import tqdm

features_cache_dir = PROJECT_ROOT / "artifacts" / "baseline_ml"
features_cache_dir.mkdir(parents=True, exist_ok=True)

cache_signature = {
    "backbone": "resnet18_imagenet1k_v1",
    "image_size": image_size,
    "train_subdir": config["data"].get("train_subdir", "train"),
    "val_subdir": config["data"].get("val_subdir", "val"),
    "num_train_samples": len(train_dataset),
    "num_val_samples": len(val_dataset),
    "class_names": list(class_names),
    "feature_dim_expected": config["model"].get("feature_dim", 512),
}
cache_hash = hashlib.sha1(
    json.dumps(cache_signature, sort_keys=True, ensure_ascii=True).encode("utf-8")
).hexdigest()[:12]
features_cache_path = features_cache_dir / f"features_cache_{cache_hash}.joblib"

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = nn.Identity()
backbone.eval().to(device)
for p in backbone.parameters():
    p.requires_grad = False

@torch.inference_mode()
def extract_features(loader, split_name: str):
    feats, labels = [], []
    split_t0 = perf_counter()

    progress = tqdm(loader, total=len(loader), desc=f"Extracting {split_name}", unit="batch")
    for images, targets in progress:
        images = images.to(device, non_blocking=True)
        outputs = backbone(images)

        feats.append(outputs.cpu().numpy())
        labels.append(targets.numpy())

    elapsed_s = perf_counter() - split_t0
    print(f"{split_name}: done in {elapsed_s/60:.1f} min ({len(loader.dataset)} images)")
    log_wandb(
        {
            "phase": f"feature_extraction_{split_name}",
            f"{split_name}_feature_minutes": elapsed_s / 60,
            f"{split_name}_feature_samples": len(loader.dataset),
        }
    )
    return np.concatenate(feats), np.concatenate(labels)

cache_loaded = False
if features_cache_path.exists():
    cache = joblib.load(features_cache_path)
    if cache.get("cache_hash") != cache_hash:
        print("Cache encontrada pero hash de configuracion no coincide. Reextrayendo features...")
    else:
        X_train = cache["X_train"]
        y_train = cache["y_train"]
        X_val = cache["X_val"]
        y_val = cache["y_val"]
        cached_classes = cache.get("class_names")
        if cached_classes is not None and list(cached_classes) != list(class_names):
            print("Cache encontrada pero class_names no coincide. Reextrayendo features...")
        else:
            cache_loaded = True
            print(f"Features cargadas desde cache: {features_cache_path}")
            log_wandb({"phase": "features_cache_hit", "feature_cache_hit": 1})

if not cache_loaded:
    X_train, y_train = extract_features(train_loader, "train")
    X_val, y_val = extract_features(val_loader, "val")
    joblib.dump(
        {
            "cache_hash": cache_hash,
            "cache_signature": cache_signature,
            "X_train": X_train,
            "y_train": y_train,
            "X_val": X_val,
            "y_val": y_val,
            "class_names": class_names,
            "image_size": image_size,
            "feature_dim": int(X_train.shape[1]),
        },
        features_cache_path,
    )
    print(f"Features guardadas en cache: {features_cache_path}")
    log_wandb({"phase": "features_cache_saved", "feature_cache_saved": 1})

print("Feature shape:", X_train.shape)
log_wandb({"phase": "features_ready", "feature_dim": int(X_train.shape[1]), "feature_cache_hash": cache_hash})

Features cargadas desde cache: c:\Users\aleja\Documents\Comillas\2º Cuatri\Análisis de Datos no Estructurados\signlanguage-classifier\artifacts\baseline_ml\features_cache_3605b175b8f8.joblib
Feature shape: (189641, 512)


## Entrenamiento de modelos sklearn

In [4]:
from time import perf_counter
import threading
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from tqdm import tqdm

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)

results = {}

rf_total_trees = 200
rf_chunk_size = 10
rf_chunks = rf_total_trees // rf_chunk_size

# 3 candidatos + bloques RF + evaluaciones y logs
total_steps = 10 + rf_chunks
pbar = tqdm(total=total_steps, desc="Baseline training", unit="step")


def eval_and_store(name, model, fit_seconds):
    val_preds = model.predict(X_val_s)
    val_acc = accuracy_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_preds, average="macro")
    results[name] = {
        "model": model,
        "val_accuracy": val_acc,
        "val_f1_macro": val_f1,
        "fit_seconds": fit_seconds,
    }
    print(f"{name}: acc={val_acc:.4f} f1_macro={val_f1:.4f} fit_s={fit_seconds:.1f}")
    log_wandb(
        {
            "phase": "candidate_eval",
            "candidate_idx": len(results),
            f"candidate/{name}/val_accuracy": val_acc,
            f"candidate/{name}/val_f1_macro": val_f1,
            f"candidate/{name}/fit_seconds": fit_seconds,
        }
    )
    return val_acc, val_f1


def fit_with_live_status(model, X, y, label: str):
    done = threading.Event()
    start = perf_counter()

    def heartbeat():
        while not done.is_set():
            elapsed = perf_counter() - start
            pbar.set_description(f"{label}: fit ({elapsed:0.0f}s)")
            time.sleep(1.0)

    thread = threading.Thread(target=heartbeat, daemon=True)
    thread.start()
    model.fit(X, y)
    done.set()
    thread.join(timeout=1.0)
    fit_s = perf_counter() - start
    return fit_s


# 1) Logistic Regression
pbar.set_description("LogReg: init")
logreg = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
pbar.update(1)

logreg_fit_s = fit_with_live_status(logreg, X_train_s, y_train, "LogReg")
pbar.update(1)

pbar.set_description("LogReg: eval")
logreg_acc, logreg_f1 = eval_and_store("logreg", logreg, logreg_fit_s)
pbar.update(1)

pbar.set_description("LogReg: log")
log_wandb(
    {
        "phase": "candidate_progress",
        "candidate_name": "logreg",
        "candidate_val_accuracy": logreg_acc,
        "candidate_val_f1_macro": logreg_f1,
    }
)
pbar.update(1)

# 2) Linear SVC
pbar.set_description("LinearSVC: init")
linear_svc = LinearSVC(class_weight="balanced")
pbar.update(1)

svc_fit_s = fit_with_live_status(linear_svc, X_train_s, y_train, "LinearSVC")
pbar.update(1)

pbar.set_description("LinearSVC: eval")
svc_acc, svc_f1 = eval_and_store("linear_svc", linear_svc, svc_fit_s)
pbar.update(1)

pbar.set_description("LinearSVC: log")
log_wandb(
    {
        "phase": "candidate_progress",
        "candidate_name": "linear_svc",
        "candidate_val_accuracy": svc_acc,
        "candidate_val_f1_macro": svc_f1,
    }
)
pbar.update(1)

# 3) Random Forest (dinamico por bloques)
pbar.set_description("RandomForest: init")
rf = RandomForestClassifier(
    n_estimators=0,
    warm_start=True,
    n_jobs=-1,
    class_weight="balanced",
    random_state=config["seed"],
)
pbar.update(1)

rf_t0 = perf_counter()
for n_trees in range(rf_chunk_size, rf_total_trees + 1, rf_chunk_size):
    rf.set_params(n_estimators=n_trees)
    rf.fit(X_train_s, y_train)
    pbar.set_description(f"RandomForest: {n_trees}/{rf_total_trees} trees")
    pbar.update(1)
rf_fit_s = perf_counter() - rf_t0

pbar.set_description("RandomForest: eval")
rf_acc, rf_f1 = eval_and_store("random_forest", rf, rf_fit_s)
pbar.update(1)

pbar.set_description("RandomForest: log")
log_wandb(
    {
        "phase": "candidate_progress",
        "candidate_name": "random_forest",
        "candidate_val_accuracy": rf_acc,
        "candidate_val_f1_macro": rf_f1,
    }
)
pbar.update(1)

pbar.set_description("Done")
pbar.close()

LinearSVC: fit (0s):  17%|█▋        | 5/30 [01:51<23:10, 55.62s/step]

logreg: acc=0.9728 f1_macro=0.9736 fit_s=111.2


RandomForest: init:  27%|██▋       | 8/30 [10:13<39:24, 107.49s/step]   c:\Users\aleja\anaconda3\envs\DL\lib\site-packages\sklearn\ensemble\_forest.py:860: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(


linear_svc: acc=0.9629 f1_macro=0.9636 fit_s=502.5


RandomForest: 10/200 trees:  33%|███▎      | 10/30 [10:26<17:57, 53.89s/step]c:\Users\aleja\anaconda3\envs\DL\lib\site-packages\sklearn\ensemble\_forest.py:860: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
RandomForest: 20/200 trees:  37%|███▋      | 11/30 [10:49<15:33, 49.13s/step]c:\Users\aleja\anaconda3\envs\DL\lib\site-packages\sklearn\ensemble\_forest.py:860: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes

random_forest: acc=0.9955 f1_macro=0.9956 fit_s=312.9


## Evaluacion final en validacion del mejor modelo

In [5]:
from sklearn.metrics import log_loss

best_name = max(results, key=lambda k: results[k]["val_f1_macro"])
best = results[best_name]["model"]
print("Best model:", best_name)

val_preds_best = best.predict(X_val_s)
val_acc_final = accuracy_score(y_val, val_preds_best)
val_f1_final = f1_score(y_val, val_preds_best, average="macro")

# "Loss" para sklearn baseline (cross-entropy en validacion)
if hasattr(best, "predict_proba"):
    val_proba_best = best.predict_proba(X_val_s)
    final_val_loss = log_loss(y_val, val_proba_best)
else:
    # Fallback para modelos sin predict_proba (p.ej. LinearSVC)
    # aproximamos con probabilidades via softmax sobre decision_function
    scores = best.decision_function(X_val_s)
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim == 1:
        scores = np.vstack([-scores, scores]).T
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    val_proba_best = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    final_val_loss = log_loss(y_val, val_proba_best)

print("Validation accuracy:", val_acc_final)
print("Validation F1 macro:", val_f1_final)
print("Validation loss (log_loss):", final_val_loss)
print(classification_report(y_val, val_preds_best, target_names=class_names))

log_wandb(
    {
        "phase": "final_validation",
        "best_model": best_name,
        "final_val_accuracy": val_acc_final,
        "final_val_f1_macro": val_f1_final,
        "final_val_loss": final_val_loss,
    }
)

Best model: random_forest
Validation accuracy: 0.995457398165028
Validation F1 macro: 0.9955891320656639
Validation loss (log_loss): 0.35079440995089567
              precision    recall  f1-score   support

           A       0.99      0.99      0.99      1269
           B       1.00      1.00      1.00      1246
           C       1.00      1.00      1.00      1222
           D       1.00      1.00      1.00      1144
           E       0.99      0.99      0.99      1162
           F       1.00      1.00      1.00      1205
           G       0.99      1.00      0.99      1177
           H       0.99      1.00      1.00      1186
           I       0.99      0.99      0.99      1193
           J       1.00      0.99      0.99      1125
           K       0.99      0.99      0.99      1181
           L       1.00      1.00      1.00      1191
           M       0.99      1.00      0.99      1185
           N       1.00      0.99      0.99      1190
           O       1.00      1.00   

## Guardado del modelo y cierre de run W&B

In [8]:
output_dir = PROJECT_ROOT / config["output"]["artifacts_dir"] / "baseline_ml"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / config["output"]["model_name"]
joblib.dump({"model": best, "scaler": scaler, "class_names": class_names, "best_name": best_name}, model_path)
print("Saved best model to", model_path)

log_wandb(
    {
        "phase": "artifact_saved",
        "artifact_baseline_saved": 1,
    }
)

ensure_wandb_run()
if run is not None:
    try:
        run.summary["best_model"] = best_name
        run.summary["val_accuracy"] = val_acc_final
        run.summary["val_f1_macro"] = val_f1_final
        run.summary["artifact_model_path"] = str(model_path)
        finish_wandb_run(run)
    except UsageError:
        print("W&B run ya estaba cerrada; se omite cierre duplicado.")

Saved best model to c:\Users\aleja\Documents\Comillas\2º Cuatri\Análisis de Datos no Estructurados\signlanguage-classifier\artifacts\baseline_ml\baseline_model.joblib


artifact_baseline_saved,▁
progress_step,▁
artifact_baseline_saved,1
artifact_model_path,c:\Users\aleja\Docum...
best_model,random_forest
phase,artifact_saved
progress_step,14
val_accuracy,0.99546
val_f1_macro,0.99559


In [7]:
!pip install nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)

   ---------------------------------------- 2/2 [nbformat]

